# 02_baseline_lookup
1. **Keep-as-is**: giữ nguyên token đầu vào
2. **Lookup table**: học từ train xem một token `raw` thường được chuẩn hóa thành token `norm` nào nhiều nhất

## 1. Import thư viện

In [1]:
import pandas as pd
import re
from collections import defaultdict, Counter
import os

## 2. Khai báo đường dẫn dữ liệu

Notebook này giả sử:
- notebook nằm trong thư mục `notebooks/`
- dữ liệu nằm trong thư mục `data/`

In [2]:
train_path = "../data/multilexnorm2026_vi_train.csv"
val_path   = "../data/multilexnorm2026_vi_validation.csv"
test_path  = "../data/multilexnorm2026_vi_test.csv"

## 3. Đọc dữ liệu

In [3]:
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

print("Train shape     :", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape      :", test_df.shape)

Train shape     : (8371, 3)
Validation shape: (1050, 3)
Test shape      : (522, 3)


## 4. Hàm parse token list

Dữ liệu trong file CSV đang ở dạng chuỗi như:

`"['t' 'ko' 'biet']"`

Ta cần đổi nó về dạng list Python:

`['t', 'ko', 'biet']`

In [4]:
def parse_token_list(text):
    text = str(text)
    tokens = re.findall(r"'([^']*)'", text)
    return tokens

## 5. Parse các cột `raw` và `norm`

In [5]:
for df in [train_df, val_df, test_df]:
    df["raw_tokens"] = df["raw"].apply(parse_token_list)
    df["norm_tokens"] = df["norm"].apply(parse_token_list)

print("Ví dụ một dòng train sau khi parse:")
print("RAW :", train_df.loc[0, "raw_tokens"])
print("NORM:", train_df.loc[0, "norm_tokens"])

Ví dụ một dòng train sau khi parse:
RAW : ['thích', 'anh', 'cá', 'mập', 'k']
NORM: ['thích', 'anh', 'cá', 'mập', 'không']


## 6. Baseline 1: Keep-as-is

Ý tưởng:
- thấy token nào thì giữ nguyên token đó
- không sửa gì cả

In [6]:
def predict_keep_as_is(raw_tokens):
    return raw_tokens[:]

## 7. Baseline 2: Lookup table

Ý tưởng:
- duyệt qua train
- đếm xem mỗi token `raw` thường đi với token `norm` nào nhiều nhất
- tạo bảng tra: `raw_token -> norm_token phổ biến nhất`

In [7]:
mapping_counter = defaultdict(Counter)

for _, row in train_df.iterrows():
    raw_tokens = row["raw_tokens"]
    norm_tokens = row["norm_tokens"]

    if len(raw_tokens) != len(norm_tokens):
        continue

    for raw_tok, norm_tok in zip(raw_tokens, norm_tokens):
        mapping_counter[raw_tok][norm_tok] += 1

lookup_table = {}
for raw_tok, counter in mapping_counter.items():
    lookup_table[raw_tok] = counter.most_common(1)[0][0]

print("Kích thước bảng tra:", len(lookup_table))

Kích thước bảng tra: 9765


## 8. Kiểm tra nhanh một vài token quen thuộc

In [8]:
for token in ["ko", "k", "đc", "dc", "e", "r", "khum", "ngta", "t", "z"]:
    print(f"{token} -> {lookup_table.get(token, token)}")

ko -> không
k -> không
đc -> được
dc -> được
e -> em
r -> rồi
khum -> không
ngta -> người ta
t -> tôi
z -> vậy


## 9. Hàm dự đoán bằng lookup table

In [9]:
def predict_lookup(raw_tokens, lookup_table):
    preds = []
    for tok in raw_tokens:
        preds.append(lookup_table.get(tok, tok))
    return preds

## 10. Hàm đánh giá trên validation

Trong notebook này mình tính 2 chỉ số dễ hiểu:

- **Token Accuracy**: tỷ lệ token dự đoán đúng
- **Sentence Accuracy**: tỷ lệ câu đúng hoàn toàn

In [10]:
def evaluate(df, predict_fn, lookup_table=None):
    total_tokens = 0
    correct_tokens = 0
    total_sentences = 0
    correct_sentences = 0

    all_predictions = []

    for _, row in df.iterrows():
        raw_tokens = row["raw_tokens"]
        gold_tokens = row["norm_tokens"]

        if lookup_table is None:
            pred_tokens = predict_fn(raw_tokens)
        else:
            pred_tokens = predict_fn(raw_tokens, lookup_table)

        all_predictions.append(pred_tokens)
        total_sentences += 1

        if len(pred_tokens) != len(gold_tokens):
            continue

        sent_ok = True
        for p, g in zip(pred_tokens, gold_tokens):
            total_tokens += 1
            if p == g:
                correct_tokens += 1
            else:
                sent_ok = False

        if sent_ok:
            correct_sentences += 1

    token_acc = correct_tokens / total_tokens if total_tokens > 0 else 0
    sent_acc = correct_sentences / total_sentences if total_sentences > 0 else 0

    return token_acc, sent_acc, all_predictions

## 11. Chạy 2 baseline trên validation

In [11]:
keep_token_acc, keep_sent_acc, keep_preds = evaluate(val_df, predict_keep_as_is)
lookup_token_acc, lookup_sent_acc, lookup_preds = evaluate(val_df, predict_lookup, lookup_table)

print("===== KẾT QUẢ VALIDATION =====")
print(f"Keep-as-is   -> Token Acc: {keep_token_acc:.4f} | Sentence Acc: {keep_sent_acc:.4f}")
print(f"Lookup table -> Token Acc: {lookup_token_acc:.4f} | Sentence Acc: {lookup_sent_acc:.4f}")

===== KẾT QUẢ VALIDATION =====
Keep-as-is   -> Token Acc: 0.8436 | Sentence Acc: 0.0000
Lookup table -> Token Acc: 0.9591 | Sentence Acc: 0.6295


## 12. So sánh trực tiếp một vài ví dụ đầu

In [12]:
for i in range(5):
    print(f"\n--- Example {i} ---")
    print("RAW   :", val_df.loc[i, "raw_tokens"])
    print("GOLD  :", val_df.loc[i, "norm_tokens"])
    print("KEEP  :", keep_preds[i])
    print("LOOKUP:", lookup_preds[i])


--- Example 0 ---
RAW   : ['Bên', 'ni', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'tê', 'đèo', 'thì', '"bẻ', 'bẻ"']
GOLD  : ['Bên', 'này', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'kia', 'đèo', 'thì', '"bẻ', 'bẻ"']
KEEP  : ['Bên', 'ni', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'tê', 'đèo', 'thì', '"bẻ', 'bẻ"']
LOOKUP: ['Bên', 'này', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'tê', 'đèo', 'thì', '"bẻ', 'bẻ"']

--- Example 1 ---
RAW   : ['2', 'mày', 'bán', 'tao', 'ngồi', 'đếm', 'xiền', 'xem', 'có', 'đủ', 'cho', '3', 'đứa', 'học', 'lại', 'khong', 'nhoa', 'nhoa']
GOLD  : ['2', 'mày', 'bán', 'tao', 'ngồi', 'đếm', 'tiền', 'xem', 'có', 'đủ', 'cho', '3', 'đứa', 'học', 'lại', 'không', 'nha', 'nha']
KEEP  : ['2', 'mày', 'bán', 'tao', 'ngồi', 'đếm', 'xiền', 'xem', 'có', 'đủ', 'cho', '3', 'đứa', 'học', 'lại', 'khong', 'nhoa', 'nhoa']
LOOKUP: ['2', 'mày', 'bán', 'tao', 'ngồi', 'đếm', 'tiền', 'xem', 'có', 'đủ', 'cho', '3', 'đứa', 'học', 'lại', 'không', 'nha', 'nha']

--- Example 2 ---
RAW   : ['Vứt', 'con', 'đấy', 'đi', 

## 13. Tìm các câu lookup dự đoán sai

Cell này rất quan trọng vì nó giúp mình nhìn cụ thể:
- câu gốc là gì
- đáp án đúng là gì
- lookup sai ở đâu

In [13]:
wrong_cases = []

for idx, row in val_df.iterrows():
    raw_tokens = row["raw_tokens"]
    gold_tokens = row["norm_tokens"]
    pred_tokens = lookup_preds[idx]

    if pred_tokens != gold_tokens:
        wrong_cases.append({
            "idx": idx,
            "raw": raw_tokens,
            "gold": gold_tokens,
            "pred": pred_tokens
        })

print("Số câu lookup dự đoán sai:", len(wrong_cases))
print("Số câu lookup dự đoán đúng hoàn toàn:", len(val_df) - len(wrong_cases))

Số câu lookup dự đoán sai: 389
Số câu lookup dự đoán đúng hoàn toàn: 661


## 14. In ra một vài câu sai để đọc bằng mắt

In [14]:
for item in wrong_cases[:10]:
    print(f"\n===== Sentence index: {item['idx']} =====")
    print("RAW :", item["raw"])
    print("GOLD:", item["gold"])
    print("PRED:", item["pred"])


===== Sentence index: 0 =====
RAW : ['Bên', 'ni', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'tê', 'đèo', 'thì', '"bẻ', 'bẻ"']
GOLD: ['Bên', 'này', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'kia', 'đèo', 'thì', '"bẻ', 'bẻ"']
PRED: ['Bên', 'này', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'tê', 'đèo', 'thì', '"bẻ', 'bẻ"']

===== Sentence index: 4 =====
RAW : ['3', 'đứa', 'm', 'có', 'v', 'k']
GOLD: ['3', 'đứa', 'm', 'có', 'vậy', 'không']
PRED: ['3', 'đứa', 'mày', 'có', 'vậy', 'không']

===== Sentence index: 7 =====
RAW : ['Đọc', 'tiêu', 'đề', 'ba', 'chấm', 'wua']
GOLD: ['Đọc', 'tiêu', 'đề', 'ba', 'chấm', 'quá']
PRED: ['Đọc', 'tiêu', 'đề', 'ba', 'chấm', 'qua']

===== Sentence index: 8 =====
RAW : ['thành', 'ra', 'đến', 'h', 'yêu', 'đương', 'vẫn', 'phải', 'giấu', 'diếm', 'chứ', 'chả', 'đc', 'thoải', 'mái', 'lắm']
GOLD: ['thành', 'ra', 'đến', 'giờ', 'yêu', 'đương', 'vẫn', 'phải', 'giấu', 'diếm', 'chứ', 'chả', 'được', 'thoải', 'mái', 'lắm']
PRED: ['thành', 'ra', 'đến', 'giờ', 'yêu', 'đương', 'vẫn', 'phải', 'gi

## 15. Đếm số token sai trong từng câu sai

Cell này giúp mình ưu tiên đọc những câu sai nặng nhất.

In [15]:
def count_token_errors(gold_tokens, pred_tokens):
    if len(gold_tokens) != len(pred_tokens):
        return max(len(gold_tokens), len(pred_tokens))
    return sum(g != p for g, p in zip(gold_tokens, pred_tokens))

wrong_cases_with_error_count = []

for item in wrong_cases:
    err_count = count_token_errors(item["gold"], item["pred"])
    new_item = item.copy()
    new_item["num_errors"] = err_count
    wrong_cases_with_error_count.append(new_item)

wrong_cases_with_error_count = sorted(
    wrong_cases_with_error_count,
    key=lambda x: x["num_errors"],
    reverse=True
)

print("Top 10 câu sai nhiều token nhất:")
for item in wrong_cases_with_error_count[:10]:
    print(f"idx={item['idx']}, num_errors={item['num_errors']}")

Top 10 câu sai nhiều token nhất:
idx=493, num_errors=10
idx=402, num_errors=6
idx=412, num_errors=6
idx=889, num_errors=6
idx=126, num_errors=5
idx=133, num_errors=5
idx=205, num_errors=5
idx=145, num_errors=4
idx=303, num_errors=4
idx=612, num_errors=4


## 16. In chi tiết các câu sai nhiều token nhất

In [16]:
for item in wrong_cases_with_error_count[:10]:
    print(f"\n===== Sentence index: {item['idx']} | num_errors={item['num_errors']} =====")
    print("RAW :", item["raw"])
    print("GOLD:", item["gold"])
    print("PRED:", item["pred"])


===== Sentence index: 493 | num_errors=10 =====
RAW : ['Tu', 'sat', 'nhung', 'khong', 'chet', 'va', 'phai', 'song', 'voi', 'hau', 'qua', 'suot', 'doi', 'cho', 'den', 'khi', 'chet', 'gia', '=))']
GOLD: ['Tự', 'sát', 'nhưng', 'không', 'chết', 'và', 'phải', 'sống', 'với', 'hậu', 'quả', 'suốt', 'đời', 'cho', 'đến', 'khi', 'chết', 'già', '=))']
PRED: ['Tu', 'sát', 'nhung', 'không', 'chết', 'va', 'phải', 'song', 'voi', 'hau', 'qua', 'suot', 'đời', 'cho', 'den', 'khi', 'chết', 'gia', '=))']

===== Sentence index: 402 | num_errors=6 =====
RAW : ['em', 'bừn', 'tữn', 'ngke', 'ăng', 'lóy', 'i', 'mò']
GOLD: ['em', 'bình', 'tĩnh', 'nghe', 'anh', 'nói', 'đi', 'mà']
PRED: ['em', 'bừn', 'tữn', 'ngke', 'ăn', 'lóy', 'đi', 'mò']

===== Sentence index: 412 | num_errors=6 =====
RAW : ['Tao', 'đi', 'xe', 'bít', 'tao', 'sợ', 'mỗi', 'ông', 'Nọi', 'tài', 'xế', 'bởi', 'dì', 'ổng', 'mà', 'bùn', 'ổng', 'hỏng', 'chịu', 'mở', 'cửa', 'xe', 'là', 'tao', 'chạy', 'như', 'điêng', 'tới', 'trạm', 'cũng', 'hỏng', 'đc', 'l

## 17. Kiểm tra một số token mơ hồ trên validation

Cell này giúp xem lookup có đang xử lý các token khó như `t`, `m`, `ko`, `k`, ... ra sao.

In [17]:
tokens_to_probe = ["t", "m", "ko", "k", "đc", "dc", "e", "ngta", "khum", "z"]

for token in tokens_to_probe:
    if token in mapping_counter:
        print(f"\n{token}:")
        print(mapping_counter[token].most_common(10))


t:
[('tôi', 933), ('tao', 152), ('tớ', 8), ('t', 5), ('tui', 4), ('thằng', 1), ('tồi', 1), ('ta', 1), ('tới', 1)]

m:
[('mày', 232), ('mình', 27), ('m', 17), ('má', 1), ('mẹ', 1), ('mi', 1), ('mọi người', 1)]

ko:
[('không', 917), ('ko', 8), ('khôngo', 1)]

k:
[('không', 911), ('k', 4)]

đc:
[('được', 527), ('đc', 8), ('đượ', 1)]

dc:
[('được', 174), ('dc', 2)]

e:
[('em', 333), ('e', 6), ('en', 2)]

ngta:
[('người ta', 159), ('ngta', 4), ('người', 1)]

khum:
[('không', 125)]

z:
[('vậy', 209), ('z', 2)]


## 18. Dự đoán cho test bằng lookup table

In [18]:
test_predictions = []

for _, row in test_df.iterrows():
    raw_tokens = row["raw_tokens"]
    pred_tokens = predict_lookup(raw_tokens, lookup_table)
    test_predictions.append(pred_tokens)

print("Đã sinh xong dự đoán cho test.")
print("Số câu test:", len(test_predictions))

Đã sinh xong dự đoán cho test.
Số câu test: 522


## 19. Lưu kết quả dự đoán test ra file

Mình lưu 2 cột phụ:
- `pred_tokens`: list token dự đoán
- `pred_str`: chuỗi ghép lại cho dễ nhìn

Lưu ý:
- file này chỉ là output nội bộ ban đầu
- chưa chắc đã là đúng format nộp cuối cùng của ban tổ chức

In [ ]:
os.makedirs("../outputs", exist_ok=True)

test_output_df = test_df.copy()
test_output_df["pred_tokens"] = test_predictions
test_output_df["pred_str"] = test_output_df["pred_tokens"].apply(lambda x: " ".join(x))

save_path = "../outputs/vi_test_predictions_lookup.csv"
test_output_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Đã lưu file:", save_path)

## 20. Lưu dữ liệu sai

In [19]:
import os
import pandas as pd

# Tạo thư mục outputs nếu chưa có
os.makedirs("../outputs", exist_ok=True)

# =========================================================
# 1) LƯU KẾT QUẢ Ở MỨC CÂU
# =========================================================
sentence_records = []

for i, (_, row) in enumerate(val_df.iterrows()):
    raw_tokens = row["raw_tokens"]
    gold_tokens = row["norm_tokens"]
    pred_tokens = lookup_preds[i]

    # Đếm số token sai và vị trí sai
    token_errors = 0
    error_positions = []

    if len(gold_tokens) == len(pred_tokens):
        for pos, (g, p) in enumerate(zip(gold_tokens, pred_tokens)):
            if g != p:
                token_errors += 1
                error_positions.append(pos)
    else:
        token_errors = max(len(gold_tokens), len(pred_tokens))
        error_positions = ["length_mismatch"]

    sentence_records.append({
        "sent_idx": i,
        "raw_tokens": str(raw_tokens),
        "gold_tokens": str(gold_tokens),
        "pred_tokens": str(pred_tokens),
        "token_errors": token_errors,
        "error_positions": str(error_positions),
        "is_sentence_correct": int(pred_tokens == gold_tokens)
    })

sentence_df = pd.DataFrame(sentence_records)

# Lưu toàn bộ kết quả mức câu
sentence_df.to_csv(
    "../outputs/val_lookup_all_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# Lưu riêng các câu sai
wrong_sentence_df = sentence_df[sentence_df["is_sentence_correct"] == 0].copy()
wrong_sentence_df = wrong_sentence_df.sort_values(by="token_errors", ascending=False)

wrong_sentence_df.to_csv(
    "../outputs/val_lookup_wrong_sentences.csv",
    index=False,
    encoding="utf-8-sig"
)

# =========================================================
# 2) LƯU KẾT QUẢ Ở MỨC TOKEN
# =========================================================
# Tạo train_vocab để đánh dấu OOV
train_vocab = set()
for tokens in train_df["raw_tokens"]:
    train_vocab.update(tokens)

token_records = []

for i, (_, row) in enumerate(val_df.iterrows()):
    raw_tokens = row["raw_tokens"]
    gold_tokens = row["norm_tokens"]
    pred_tokens = lookup_preds[i]

    if len(raw_tokens) != len(gold_tokens) or len(gold_tokens) != len(pred_tokens):
        continue

    for pos, (raw_tok, gold_tok, pred_tok) in enumerate(zip(raw_tokens, gold_tokens, pred_tokens)):
        token_records.append({
            "sent_idx": i,
            "token_pos": pos,
            "raw_token": raw_tok,
            "gold_token": gold_tok,
            "pred_token": pred_tok,
            "is_correct": int(gold_tok == pred_tok),
            "needs_normalization": int(raw_tok != gold_tok),
            "lookup_changed": int(raw_tok != pred_tok),
            "is_oov_vs_train": int(raw_tok not in train_vocab)
        })

token_df = pd.DataFrame(token_records)

# Lưu toàn bộ mức token
token_df.to_csv(
    "../outputs/val_lookup_token_level_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# Lưu riêng token sai
wrong_token_df = token_df[token_df["is_correct"] == 0].copy()

wrong_token_df.to_csv(
    "../outputs/val_lookup_wrong_tokens.csv",
    index=False,
    encoding="utf-8-sig"
)

# =========================================================
# 3) IN THÔNG BÁO
# =========================================================
print("Đã lưu xong:")
print("- ../outputs/val_lookup_all_results.csv")
print("- ../outputs/val_lookup_wrong_sentences.csv")
print("- ../outputs/val_lookup_token_level_results.csv")
print("- ../outputs/val_lookup_wrong_tokens.csv")

print("\nSố câu sai:", len(wrong_sentence_df))
print("Số token sai:", len(wrong_token_df))

Đã lưu xong:
- ../outputs/val_lookup_all_results.csv
- ../outputs/val_lookup_wrong_sentences.csv
- ../outputs/val_lookup_token_level_results.csv
- ../outputs/val_lookup_wrong_tokens.csv

Số câu sai: 389
Số token sai: 558


In [20]:
import pandas as pd

# Đọc file token sai đã lưu
wrong_token_df = pd.read_csv("../outputs/val_lookup_wrong_tokens.csv")

print("Số token sai:", len(wrong_token_df))
print("\n5 dòng đầu:")
display(wrong_token_df.head())

# =========================================================
# 1) Token raw nào bị sai nhiều nhất
# =========================================================
print("\n=== Top raw tokens bị lookup đoán sai nhiều nhất ===")
raw_error_counts = (
    wrong_token_df["raw_token"]
    .value_counts()
    .reset_index()
)
raw_error_counts.columns = ["raw_token", "error_count"]
display(raw_error_counts.head(20))

# =========================================================
# 2) Các cặp sai phổ biến nhất: raw -> gold và raw -> pred
# =========================================================
print("\n=== Top các lỗi phổ biến nhất ===")
error_pattern_df = (
    wrong_token_df
    .groupby(["raw_token", "gold_token", "pred_token"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
display(error_pattern_df.head(30))

# =========================================================
# 3) Trong các token sai, token nào là OOV nhiều nhất
# =========================================================
print("\n=== Số token sai là OOV ===")
oov_stats = (
    wrong_token_df["is_oov_vs_train"]
    .value_counts()
    .sort_index()
    .reset_index()
)
oov_stats.columns = ["is_oov_vs_train", "count"]
display(oov_stats)

# =========================================================
# 4) Những token cần normalize mà lookup vẫn sai nhiều
# =========================================================
print("\n=== Top token cần normalize nhưng lookup làm sai ===")
need_norm_wrong_df = wrong_token_df[wrong_token_df["needs_normalization"] == 1].copy()

need_norm_counts = (
    need_norm_wrong_df["raw_token"]
    .value_counts()
    .reset_index()
)
need_norm_counts.columns = ["raw_token", "wrong_count"]
display(need_norm_counts.head(20))

# =========================================================
# 5) Top câu có nhiều token sai nhất
# =========================================================
print("\n=== Top câu có nhiều token sai nhất ===")
sent_error_counts = (
    wrong_token_df["sent_idx"]
    .value_counts()
    .reset_index()
)
sent_error_counts.columns = ["sent_idx", "num_wrong_tokens"]
display(sent_error_counts.head(20))

Số token sai: 558

5 dòng đầu:


,sent_idx,token_pos,raw_token,gold_token,pred_token,is_correct,needs_normalization,lookup_changed,is_oov_vs_train
0,0,7,tê,kia,tê,0,1,0,0
1,4,2,m,m,mày,0,0,1,0
2,7,5,wua,quá,qua,0,1,1,0
3,8,9,diếm,diếm,giếm,0,0,1,0
4,9,0,Dơ,Giơ,Dơ,0,1,0,1



=== Top raw tokens bị lookup đoán sai nhiều nhất ===


,raw_token,error_count
0,t,17
1,m,15
2,c,12
3,tr,4
4,dị,4
5,ma,4
6,hok,3
7,th,3
8,thiệt,3
9,hỏng,3



=== Top các lỗi phổ biến nhất ===


...

,is_oov_vs_train,count
0,0,322
1,1,236



=== Top token cần normalize nhưng lookup làm sai ===


,raw_token,wrong_count
0,t,16
1,c,11
2,m,9
3,hok,3
4,th,3
5,tr,3
6,thiệt,3
7,nà,3
8,nc,3
9,bh,3



=== Top câu có nhiều token sai nhất ===


,sent_idx,num_wrong_tokens
0,493,10
1,402,6
2,412,6
3,889,6
4,126,5
5,133,5
6,205,5
7,145,4
8,303,4
9,612,4
